In [ ]:
library(Seurat)
library(ggplot2)
library(pheatmap)
library(dplyr)
library(RColorBrewer)
library(reshape2)
library(clustree)
getwd()
dir.create("figures_10xPBMC_pseudobulk")
dir.create("data_10xPBMC")
dataset_id <- "10xPBMC"
sample <- "pbmc8k"
mode <- "pseudobulk"

colorPBMC <- "#81B29A"

PBMCcelltypeColors <- c("B"="#91bcca",
                "CD14pos_Monocytes"="#3d405b",
                "CD8A"="#f2cc8f",
                "Dendritic"="#c492a7",
                "FCGR3Apos_Monocytes"="#e78063",
                "Megakaryocytes"="#c4b3ab",
                "Memory_CD4_T"="#81b29a",
                "Naive_CD4_T"="#4d7362", 
                "NK"= "#88535a")

In [ ]:

path <- paste0("/mnt/TEresults2/snakemake_results/results/stellarscope_out/", dataset_id, "/", sample, "/", mode, "/")


TEmatrix <- Seurat::ReadMtx(mtx = paste0(path, sample, "_", mode, "-TE_counts.mtx"), 
                              cells = paste0(path, sample, "_", mode, "-barcodes.tsv"), 
                              features = paste0(path, sample, "_", mode, "-features.tsv"), 
                              feature.column = 1) # read matrix
TEmatrix <- TEmatrix[setdiff(rownames(TEmatrix),"__no_feature"),]
nCells <- ncol(TEmatrix)
thrMinCells <- round(nCells * 0.05)

# create Seurat object with shallow filtering of TEs expressed in at least 50 cells and cells expressing at least 50 TEs 
objTE_allCB <- Seurat::CreateSeuratObject(TEmatrix, project = paste0("PBMC_",mode), 
                          min.cells = thrMinCells, min.features = 50)

STAR_path <- paste0("/mnt/TEresults/snakemake_results/results/STARoutdir/", dataset_id, "/", sample, "/best_Solo.out/Gene")
filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo


objTE_stellarscope <- objTE_allCB[,filteredBarcodes] # keep only filtered barcodes

objTE_stellarscope

In [ ]:
annotation_stellarscope <- read.table("annotation/annotation_stellarscope_hg38.tsv") # generated with annotation_scripts/create_annotations_hg38.Rmd
head(annotation_stellarscope)

In [ ]:
# Remove some classes of TEs

classesToExclude <- c("Other", "Satellite", "Unknown", "RNA")

stellarscopeTEs <- Features(objTE_stellarscope)

TEsToKeep <- annotation_stellarscope[! annotation_stellarscope$class %in% classesToExclude, ]$stellarscopeID

length(setdiff(stellarscopeTEs, TEsToKeep))/ length(stellarscopeTEs) * 100 # percentage removed

objTE_stellarscope <- objTE_stellarscope[intersect(stellarscopeTEs, TEsToKeep),]

In [ ]:
feature_metadata <- annotation_stellarscope[match(Features(objTE_stellarscope), annotation_stellarscope$stellarscopeID),]

head(feature_metadata)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)


objTE_stellarscope@meta.data$nCount_TE <- objTE_stellarscope@meta.data$nCount_RNA 
objTE_stellarscope@meta.data$nFeature_TE <- objTE_stellarscope@meta.data$nFeature_RNA 
# Visualize QC metrics as a violin plot
VlnPlot(objTE_stellarscope, features = c("nCount_TE"), ncol = 1, 
        cols = colorPBMC, pt.size = 0) + theme(text=element_text(size=17))

ggsave(paste0("figures_",dataset_id,"/nCountTE_violin_stellarscope_",mode,".png"), device='png',dpi=600)
ggsave(paste0("figures_",dataset_id,"/nCountTE_violin_stellarscope_",mode,".pdf"), device='pdf')

VlnPlot(objTE_stellarscope, features = c("nFeature_TE"), ncol = 1, 
        cols = colorPBMC, pt.size = 0) + theme(text=element_text(size=17))
ggsave(paste0("figures_",dataset_id,"/nFeatureTE_violin_stellarscope_",mode,".png"), device='png',dpi=600)
ggsave(paste0("figures_",dataset_id,"/nFeatureTE_violin_stellarscope_",mode,".pdf"), device='pdf')


In [ ]:

objTE_stellarscope <- JoinLayers(objTE_stellarscope)

objTE_stellarscope <- NormalizeData(objTE_stellarscope, normalization.method = "LogNormalize", scale.factor = 10000)

objTE_stellarscope <- FindVariableFeatures(objTE_stellarscope, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objTE_stellarscope), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objTE_stellarscope)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:

gc()
all.genes <- rownames(objTE_stellarscope)
objTE_stellarscope <- ScaleData(objTE_stellarscope) # on hvgs


In [ ]:

objTE_stellarscope <- RunPCA(objTE_stellarscope, features = VariableFeatures(object = objTE_stellarscope))

DimPlot(objTE_stellarscope, reduction = "pca") + NoLegend()

ElbowPlot(objTE_stellarscope)


In [ ]:
objTE_stellarscope <- FindNeighbors(objTE_stellarscope, dims = 1:13, k.param = 20)
objTE_stellarscope <- FindClusters(objTE_stellarscope, resolution = 1.2)
objTE_stellarscope <- RunUMAP(objTE_stellarscope, dims = 1:13)
DimPlot(objTE_stellarscope, reduction = "umap")


In [ ]:
FeaturePlot(objTE_stellarscope, reduction = "umap", features = "nCount_TE", pt.size = 0.5) + 
  theme_void() +
  theme(text=element_text(size=20))

In [ ]:
celltypes <- read.table("/mnt/TEdata/TEbenchmarking/data/whitelists/celltype_annotation_sub_min.tsv")
table(Cells(objTE_stellarscope) %in% celltypes$V1)

objTE_stellarscope$celltype <- celltypes$V2[match(Cells(objTE_stellarscope), celltypes$V1)]
table(objTE_stellarscope$celltype)

In [ ]:
options(repr.plot.width=7, repr.plot.height=7)

DimPlot(objTE_stellarscope, reduction = "umap", group.by = "celltype",
        cols=PBMCcelltypeColors,
        shuffle=T, pt.size = 1) + 
  theme_void() +
  theme(text=element_text(size=20)) 
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_celltypes_stellarscope_",mode,".png"), device = "png")
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_celltypes_stellarscope_",mode,".pdf"), device = "pdf")

In [ ]:

saveRDS(objTE_stellarscope, paste0("data_", dataset_id, "/stellarscope_", dataset_id, "_", mode, "seuratObj.RDS"))

In [ ]:
options(repr.plot.width=17, repr.plot.height=7)

# Add cluster information from objTE_clustered to objGenes_clustered
objTE_stellarscope$clusters.0.2 <- objTE_stellarscope$seurat_clusters
objTE_stellarscope$clusters.0.1 <- objTE_stellarscope$celltype

concordanceTable <- table(objTE_stellarscope$seurat_clusters, objTE_stellarscope$celltype)



pheatmap(concordanceTable, display_numbers = T, color = brewer.pal(9,'Blues')[1:6],
         border_color = NA, number_format = "%.0f", cellwidth = 25, cellheight = 25)

# Generate the clustree plot
clustree(objTE_stellarscope, prefix = "clusters.", node_text_angle=20, node_text_size=4) + theme(text=element_text(size=15)) + 
  scale_color_manual(values=alpha(c("#81B29A","#F2CC8F"), 0.6)) + scale_size(range = c(3,20)) +
  guides(colour = FALSE) 

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_stellarscope$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

df$cluster <- factor(df$cluster, levels=c(0,1:length(unique(df$cluster))))


ggplot(df, aes(x=cluster, y=percentage, fill=celltype)) + 
  geom_col() + 
  scale_fill_manual(values = PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave(paste0("figures_10xPBMC/clusterIdentity_barplot_Stellarscope,",mode,".pdf"), width=8, height=3)
